In [1]:
import spacy
import locations_ner as ner


nlp_small = spacy.load("de_core_news_sm", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"])
nlp_medium = spacy.load("de_core_news_md", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"])
nlp_large = spacy.load("de_core_news_lg", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"])


In [2]:
possible_labels = nlp_small.get_pipe('ner').labels
possible_labels

('LOC', 'MISC', 'ORG', 'PER')

In [ ]:
bsp_texte = ['Mark hat eine Knie OP in berlin',
            'wo finde ich einen arzt für eine lungenentzündung nahe augsburg',
            'Harzinfarkt Hambuurg',
            'Herzinfarkt Hamburg',
            'Carcinoma hamburg',
            'Vladivostock',
            'Zahnarzt in Beijing',
            'Krankheit nahe Stadthausen',
            'Hogwarts',
            "muenchen",]



In [4]:
models= [nlp_small, nlp_medium, nlp_large]

In [5]:
for model in models:
    print(ner.extract_locations(bsp_texte, model))

['Mark', 'berlin', 'augsburg', 'Beijing', 'Stadthausen']
['berlin', 'augsburg', 'Vladivostock', 'Beijing']
['berlin', 'augsburg', 'Vladivostock', 'Beijing', 'Stadthausen', 'Hogwarts']


In [6]:
for doc in [nlp_small(text) for text in bsp_texte]:
    for ent in doc.ents:
        print( f"erkennt :{ent.text} als: {ent.label_}")

erkennt :Mark als: LOC
erkennt :berlin als: LOC
erkennt :augsburg als: LOC
erkennt :Harzinfarkt Hambuurg als: PER
erkennt :Herzinfarkt Hamburg als: ORG
erkennt :Carcinoma als: PER
erkennt :Vladivostock als: PER
erkennt :Beijing als: LOC
erkennt :Stadthausen als: LOC
erkennt :Hogwarts als: PER


In [7]:
for doc in [nlp_medium(text) for text in bsp_texte]:
    for ent in doc.ents:
        print( f"erkennt :{ent.text} als: {ent.label_}")

erkennt :Mark als: PER
erkennt :berlin als: LOC
erkennt :augsburg als: LOC
erkennt :Harzinfarkt Hambuurg als: PER
erkennt :Herzinfarkt Hamburg als: ORG
erkennt :Carcinoma hamburg als: ORG
erkennt :Vladivostock als: LOC
erkennt :Beijing als: LOC
erkennt :Hogwarts als: PER


In [8]:
for doc in [nlp_large(text) for text in bsp_texte]:
    for ent in doc.ents:
        print( f"erkennt :{ent.text} als: {ent.label_}")

erkennt :Mark als: MISC
erkennt :berlin als: LOC
erkennt :augsburg als: LOC
erkennt :Harzinfarkt Hambuurg als: PER
erkennt :Herzinfarkt Hamburg als: MISC
erkennt :Carcinoma hamburg als: ORG
erkennt :Vladivostock als: LOC
erkennt :Beijing als: LOC
erkennt :Stadthausen als: LOC
erkennt :Hogwarts als: LOC


In [9]:
from flair.data import Sentence
from flair.models import SequenceTagger


c:\Users\SebastianHümpfner\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
tagger = SequenceTagger.load("flair/ner-german-large")

bsp_sentences = [Sentence(text) for text in bsp_texte]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\SebastianHümpfner\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SebastianHümpfner\.flair\models\ner-german-large\models--flair--ner-german-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. 

2026-09-09 20:43:50,374 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>


In [15]:
for sentence in bsp_sentences:
    tagger.predict(sentence)
    for ent in sentence.get_spans('ner'):
        print(ent)

Span[0:1]: "Mark" → PER (1.0000)
Span[6:7]: "berlin" → LOC (1.0000)
Span[9:10]: "augsburg" → LOC (1.0000)
Span[1:2]: "Hamburg" → LOC (1.0000)
Span[1:2]: "hamburg" → LOC (1.0000)
Span[0:1]: "Vladivostock" → LOC (1.0000)
Span[2:3]: "Beijing" → LOC (1.0000)
Span[2:3]: "Stadthausen" → LOC (1.0000)
Span[0:1]: "Hogwarts" → ORG (0.9970)


In [25]:
response = ner.load_coordinates("München")

In [26]:
print(response)

<Response [403]>


In [24]:
import importlib

importlib.reload(ner)

<module 'locations_ner' from 'd:\\15 - Arbeit\\Projekte\\klinik-atlas\\klinikatlas-search\\notebooks\\locations_ner.py'>

In [3]:
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_codes = api.fileadmin_json_icd_codes_json_get()
    ops_codes = api.fileadmin_json_ops_codes_json_get()
    locations = api.fileadmin_json_locations_json_get()


In [10]:
cities = [location["city"].lower() for location in locations]

In [11]:
cities_without_duplicates = set(cities)

In [12]:
len(cities_without_duplicates)

992

In [13]:
cities_without_duplicates

{'aachen',
 'aalen, württemberg',
 'achern, baden',
 'achim bei bremen',
 'adorf/vogtland',
 'ahaus',
 'ahlen, westfalen',
 'aichach',
 'albstadt, württemberg',
 'alfeld(leine)',
 'allensbach',
 'alsfeld',
 'altdorf bei nürnberg',
 'altenburg, thüringen',
 'altenkirchen(westerwald)',
 'altentreptow',
 'altötting',
 'alzenau in unterfranken',
 'alzey',
 'amberg, oberpfalz',
 'andernach',
 'angermünde',
 'anklam',
 'ankum',
 'annaberg-buchholz',
 'ansbach',
 'ansbach, mittelfranken',
 'apolda',
 'arnsberg, westfalen',
 'arnsdorf bei dresden',
 'arnstadt',
 'asbach, westerwald',
 'aschaffenburg',
 'aschau i. chiemgau',
 'aschersleben, sachsen-anhalt',
 'attendorn',
 'aue, sachsen',
 'augsburg, bayern',
 'aurich, ostfriesland',
 'bad abbach',
 'bad aibling',
 'bad arolsen',
 'bad bellingen, baden',
 'bad belzig',
 'bad bentheim',
 'bad bergzabern',
 'bad berka',
 'bad berleburg',
 'bad bertrich',
 'bad bevensen',
 'bad bramstedt',
 'bad brückenau',
 'bad camberg',
 'bad driburg',
 'bad düb